# Notebook 11 — Agenti di Pianificazione e Sistemi Multi-Agente

**Settimana:** 11
**Blocco:** 5.3 — Orchestrazione avanzata
**Tempo stimato:** 4-5 ore

## Cosa costruirai

- Un pianificatore che scompone un compito complesso in sottocompiti strutturati
- Un esecutore che esegue il piano passo dopo passo con ripianificazione in caso di fallimenti
- Un sistema multi-agente Supervisor + Workers per l'analisi di dominio
- **Sfida finale:** pipeline di intelligence competitiva che combina ricerca, analisi e redazione

## Prerequisiti

Settimane 9 e 10 completate. Strumenti del notebook 09 disponibili.

```bash
pip install openai requests python-dotenv
```

---
## Cella 0 — Setup e strumenti di base

In [ ]:
import os
import json
import math
import uuid
import time
import datetime
import requests
from dataclasses import dataclass, field, asdict
from typing import Callable, Any
from enum import Enum
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
cliente = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODELO  = "gpt-4o-mini"

# ── Helper LLM ───────────────────────────────────────────────────────────────

def completar(
    mensajes: list[dict],
    system: str | None = None,
    temperature: float = 0.2,
    max_tokens: int = 1024
) -> str:
    msgs = ([{"role": "system", "content": system}] if system else []) + mensajes
    r = cliente.chat.completions.create(
        model=MODELO, messages=msgs,
        temperature=temperature, max_tokens=max_tokens
    )
    return r.choices[0].message.content

def completar_json(mensajes: list[dict], system: str | None = None) -> dict:
    msgs = ([{"role": "system", "content": system}] if system else []) + mensajes
    r = cliente.chat.completions.create(
        model=MODELO, messages=msgs, temperature=0.0,
        response_format={"type": "json_object"}
    )
    return json.loads(r.choices[0].message.content)

# ── Strumenti (dal notebook 09) ──────────────────────────────────────────────

def buscar_wikipedia(query: str, sentences: int = 4) -> str:
    url    = "https://en.wikipedia.org/w/api.php"
    params = {"action": "query", "format": "json", "list": "search",
               "srsearch": query, "srlimit": 1}
    try:
        res    = requests.get(url, params=params, timeout=8).json()
        if not res["query"]["search"]:
            return f"No results for '{query}'."
        titulo = res["query"]["search"][0]["title"]
        p2     = {"action": "query", "format": "json", "titles": titulo,
                  "prop": "extracts", "exsentences": sentences,
                  "exintro": True, "explaintext": True}
        pag    = requests.get(url, params=p2, timeout=8).json()
        texto  = list(pag["query"]["pages"].values())[0].get("extract", "")
        return f"[Wikipedia: {titulo}]\n{texto}"
    except Exception as e:
        return f"Wikipedia error: {e}"

def calcular(expresion: str) -> str:
    permitidos = {"__builtins__": {}, "abs": abs, "round": round,
                  "sqrt": math.sqrt, "log": math.log, "pi": math.pi}
    try:
        return f"{expresion} = {eval(expresion, permitidos)}"
    except Exception as e:
        return f"Calc error: {e}"

def obtener_fecha() -> str:
    return datetime.datetime.utcnow().strftime("%Y-%m-%d")

# Registro degli strumenti disponibili per gli agenti
HERRAMIENTAS: dict[str, Callable] = {
    "buscar_wikipedia": buscar_wikipedia,
    "calcular":         calcular,
    "obtener_fecha":    lambda: obtener_fecha(),
}

print("Setup completato.")
print(f"Strumenti disponibili: {list(HERRAMIENTAS.keys())}")

---
## Cella 1 — Pianificatore: scompone un compito in sottocompiti

Il pianificatore riceve un compito complesso e genera un piano strutturato.
Il piano include le dipendenze tra i passi per rilevare la parallelizzazione.

In [ ]:
class EstadoPaso(str, Enum):
    PENDIENTE  = "pendiente"
    EN_CURSO   = "en_curso"
    COMPLETADO = "completado"
    FALLIDO    = "fallido"
    SALTADO    = "saltado"


@dataclass
class Subtarea:
    id:           str
    descripcion:  str
    herramienta:  str | None        # nome dello strumento, o None se è LLM diretto
    argumentos:   dict | None       # argomenti dello strumento
    depende_de:   list[str]         # ID dei passi che devono essere completati prima
    resultado:    str | None = None
    error:        str | None = None
    estado:       EstadoPaso = EstadoPaso.PENDIENTE


@dataclass
class Plan:
    id:       str
    objetivo: str
    pasos:    list[Subtarea]
    estado:   str = "borrador"      # "borrador" | "en_ejecucion" | "completado"

    def pasos_listos(self) -> list[Subtarea]:
        """Passi le cui dipendenze sono già state completate."""
        completados = {p.id for p in self.pasos if p.estado == EstadoPaso.COMPLETADO}
        return [
            p for p in self.pasos
            if p.estado == EstadoPaso.PENDIENTE
            and all(dep in completados for dep in p.depende_de)
        ]

    def esta_completo(self) -> bool:
        return all(p.estado in (EstadoPaso.COMPLETADO, EstadoPaso.SALTADO) for p in self.pasos)

    def mostrar(self) -> None:
        iconos = {
            EstadoPaso.PENDIENTE:  "○",
            EstadoPaso.EN_CURSO:   "◎",
            EstadoPaso.COMPLETADO: "●",
            EstadoPaso.FALLIDO:    "✗",
            EstadoPaso.SALTADO:    "—",
        }
        print(f"\nPiano: {self.id} | Stato: {self.estado}")
        print(f"Obiettivo: {self.objetivo}")
        print("-" * 55)
        for p in self.pasos:
            icono = iconos.get(p.estado, "?")
            herr  = f"[{p.herramienta}]" if p.herramienta else "[LLM]"
            deps  = f" ← {p.depende_de}" if p.depende_de else ""
            print(f"  {icono} {p.id}: {p.descripcion[:50]}{deps}")
            if p.resultado:
                print(f"      → {p.resultado[:70]}...")


SYSTEM_PLANIFICADOR = """You are a task planner for an AI agent.
Given a complex objective, decompose it into concrete, executable steps.
Each step should either:
  - Call one specific tool (buscar_wikipedia, calcular, obtener_fecha)
  - Or be a synthesis step (herramienta: null) done directly by the LLM

Rules:
- Maximum 6 steps per plan
- Each step must be independently executable
- Mark dependencies accurately: step B depends on A only if B NEEDS A's result
- Always end with a synthesis step that integrates all findings"""


def crear_plan(objetivo: str) -> Plan:
    """
    Genera un piano strutturato per l'obiettivo dato.
    L'LLM decide quanti passi e quali strumenti usare.
    """
    herramientas_desc = json.dumps([
        {"nombre": "buscar_wikipedia", "para": "factual background information",
         "argumentos": {"query": "string", "sentences": "int (1-6)"}},
        {"nombre": "calcular",         "para": "mathematical expressions",
         "argumentos": {"expresion": "valid Python math expression"}},
        {"nombre": "obtener_fecha",    "para": "current date",
         "argumentos": {}},
    ])

    prompt = f"""Create a step-by-step plan to accomplish this objective:
OBJECTIVE: {objetivo}

Available tools:
{herramientas_desc}

Return JSON:
{{
  "pasos": [
    {{
      "id": "step_1",
      "descripcion": "What this step does",
      "herramienta": "buscar_wikipedia" | "calcular" | "obtener_fecha" | null,
      "argumentos": {{...}} | null,
      "depende_de": []
    }}
  ]
}}"""

    datos = completar_json([{"role": "user", "content": prompt}], system=SYSTEM_PLANIFICADOR)

    pasos = [
        Subtarea(
            id=p["id"],
            descripcion=p["descripcion"],
            herramienta=p.get("herramienta"),
            argumentos=p.get("argumentos") or {},
            depende_de=p.get("depende_de", [])
        )
        for p in datos["pasos"]
    ]

    return Plan(
        id=f"plan_{uuid.uuid4().hex[:6]}",
        objetivo=objetivo,
        pasos=pasos
    )


# ── Genera e mostra piani per obiettivi diversi ───────────────────────────────
OBJETIVOS = [
    "Compare PostgreSQL and MongoDB: history, use cases, and key differences.",
    "Calculate how many days ago Python was first released (February 20, 1991) and summarize its history.",
]

planes_generados = []
for obj in OBJETIVOS:
    plan = crear_plan(obj)
    plan.mostrar()
    planes_generados.append(plan)

    # Mostra quali passi possono essere eseguiti in parallelo
    listos = plan.pasos_listos()
    print(f"\n  Passi eseguibili ora ({len(listos)}): {[p.id for p in listos]}")

---
## Cella 2 — Esecutore: esegue il piano con ripianificazione

L'esecutore percorre il piano passo dopo passo.
Se un passo fallisce, cerca di ripianificare il resto invece di interrompere tutto.

In [ ]:
SYSTEM_REPLANIFICADOR = """You are replanning an AI agent's steps after a failure.
Given the failed step and the remaining plan, suggest how to proceed.
Be practical: sometimes the best option is to skip and continue with available information."""

SYSTEM_SINTETIZADOR = """You are synthesizing the results of a multi-step research plan.
Integrate all findings into a coherent, well-structured response.
Be factual and cite which step provided each piece of information.
If some steps failed or were skipped, note what information is missing."""


def ejecutar_paso(paso: Subtarea, contexto: dict[str, str]) -> str:
    """
    Esegue un singolo passo del piano.
    Se ha uno strumento: lo chiama.
    Se no: usa l'LLM con i risultati precedenti come contesto.
    """
    if paso.herramienta:
        if paso.herramienta not in HERRAMIENTAS:
            return f"Errore: lo strumento '{paso.herramienta}' non è disponibile."
        argumentos = paso.argumentos or {}
        # Sostituisce i riferimenti ai risultati precedenti negli argomenti
        argumentos_resueltos = {
            k: v.format(**contexto) if isinstance(v, str) else v
            for k, v in argumentos.items()
        }
        try:
            return HERRAMIENTAS[paso.herramienta](**argumentos_resueltos)
        except TypeError:
            return HERRAMIENTAS[paso.herramienta](**argumentos)
    else:
        # Passo di sintesi: l'LLM integra i risultati precedenti
        contexto_str = "\n".join(
            f"[{id_paso}] {resultado[:300]}" for id_paso, resultado in contexto.items()
        )
        return completar(
            [{"role": "user", "content": f"Task: {paso.descripcion}\n\nAvailable context:\n{contexto_str}"}],
            system=SYSTEM_SINTETIZADOR,
            max_tokens=600
        )


def replanificar(plan: Plan, paso_fallido: Subtarea, error: str) -> list[Subtarea]:
    """
    Suggerisce come modificare i passi rimanenti dopo un fallimento.
    Restituisce i passi rivisti (può saltare quello fallito o aggiungere alternative).
    """
    completados = [p for p in plan.pasos if p.estado == EstadoPaso.COMPLETADO]
    pendientes  = [p for p in plan.pasos if p.estado == EstadoPaso.PENDIENTE]

    prompt = f"""A step failed. Suggest how to revise the remaining plan.

Objective: {plan.objetivo}
Failed step: {paso_fallido.descripcion}
Error: {error}

Completed steps: {[p.id for p in completados]}
Remaining steps: {[p.descripcion for p in pendientes]}

Options: skip the failed step, find an alternative approach, or simplify remaining steps.
Return JSON: {{"pasos_revisados": [{{"id": "step_X", "descripcion": "...", "herramienta": "..."|null, "argumentos": {{}}|null, "depende_de": []}}]}}"""

    try:
        datos   = completar_json([{"role": "user", "content": prompt}], system=SYSTEM_REPLANIFICADOR)
        return [
            Subtarea(
                id=p["id"] + "_r",   # suffisso _r per indicare che è stato ripianificato
                descripcion=p["descripcion"],
                herramienta=p.get("herramienta"),
                argumentos=p.get("argumentos") or {},
                depende_de=p.get("depende_de", [])
            )
            for p in datos.get("pasos_revisados", [])
        ]
    except Exception:
        # Se la ripianificazione fallisce, salta semplicemente il passo
        return pendientes


def ejecutar_plan(
    plan: Plan,
    max_reintentos: int = 1,
    verbose: bool = True
) -> tuple[Plan, str]:
    """
    Esegue il piano completo passo dopo passo.
    Gestisce i fallimenti con la ripianificazione.
    Restituisce (plan_ejecutado, respuesta_final).
    """
    plan.estado    = "en_ejecucion"
    contexto       = {}    # {id_paso: resultado}
    t_inicio_total = time.perf_counter()

    if verbose:
        print(f"\nEsecuzione del piano '{plan.id}'")
        print(f"Obiettivo: {plan.objetivo}")
        print(f"Passi: {len(plan.pasos)}")

    # Il loop principale: finché ci sono passi pronti da eseguire
    iteracion = 0
    while not plan.esta_completo() and iteracion < 20:
        iteracion += 1
        listos = plan.pasos_listos()
        if not listos:
            if verbose:
                print("  Nessun passo eseguibile. Possibile deadlock nelle dipendenze.")
            break

        for paso in listos:
            paso.estado = EstadoPaso.EN_CURSO
            t_paso      = time.perf_counter()

            if verbose:
                herr = f"[{paso.herramienta}]" if paso.herramienta else "[LLM]"
                print(f"\n  → {paso.id} {herr}: {paso.descripcion[:55]}")

            intento = 0
            while intento <= max_reintentos:
                try:
                    resultado    = ejecutar_paso(paso, contexto)
                    paso.resultado = resultado
                    paso.estado  = EstadoPaso.COMPLETADO
                    contexto[paso.id] = resultado

                    latencia = (time.perf_counter() - t_paso) * 1000
                    if verbose:
                        print(f"     ✓ {resultado[:90]}... ({latencia:.0f}ms)")
                    break

                except Exception as e:
                    intento += 1
                    if intento > max_reintentos:
                        paso.estado = EstadoPaso.FALLIDO
                        paso.error  = str(e)
                        if verbose:
                            print(f"     ✗ Fallito: {e}")
                        # Ripianifica il resto
                        nuevos_pasos = replanificar(plan, paso, str(e))
                        # Sostituisce i pendenti con quelli ripianificati
                        plan.pasos = (
                            [p for p in plan.pasos if p.estado != EstadoPaso.PENDIENTE]
                            + nuevos_pasos
                        )
                        if verbose:
                            print(f"     Ripianificato: {len(nuevos_pasos)} nuovi passi")

    # Sintesi finale
    resultados_str = "\n\n".join(
        f"[{p.id}: {p.descripcion[:40]}]\n{p.resultado or 'Nessun risultato.'}"
        for p in plan.pasos if p.resultado
    )

    respuesta_final = completar(
        [{"role": "user", "content": f"Objective: {plan.objetivo}\n\nResults:\n{resultados_str}\n\nProvide a clear, integrated answer."}],
        system=SYSTEM_SINTETIZADOR,
        max_tokens=800,
        temperature=0.1
    )

    plan.estado = "completado"
    latencia_total = (time.perf_counter() - t_inicio_total) * 1000

    if verbose:
        completados = sum(1 for p in plan.pasos if p.estado == EstadoPaso.COMPLETADO)
        fallidos    = sum(1 for p in plan.pasos if p.estado == EstadoPaso.FALLIDO)
        print(f"\n{'='*55}")
        print(f"Piano completato in {latencia_total:.0f}ms")
        print(f"Passi: {completados} OK, {fallidos} falliti")

    return plan, respuesta_final


# ── Esegue i piani generati nella cella precedente ─────────────────────────────
plan_a_ejecutar = planes_generados[0]   # PostgreSQL vs MongoDB
plan_ejecutado, respuesta = ejecutar_plan(plan_a_ejecutar, verbose=True)

plan_ejecutado.mostrar()
print(f"\nRISPOSTA FINALE:\n{'='*55}")
print(respuesta)

---
## Cella 3 — Sistema multi-agente: Supervisor + Workers

Il supervisore delega sottocompiti ad agenti specializzati.
Ogni worker ha un ruolo e un insieme di strumenti diverso.

In [ ]:
@dataclass
class Delegacion:
    """Registro di una delega dal supervisore a un worker."""
    ronda:      int
    worker:     str
    subtarea:   str
    resultado:  str
    latencia_ms: float


# ── Definizione dei worker ────────────────────────────────────────────────────

def worker_investigador(subtarea: str) -> str:
    """
    Worker specializzato nella ricerca di informazioni fattuali.
    Usa Wikipedia come fonte principale.
    """
    system = """You are a research specialist. Your job is to find accurate factual information.
Use the buscar_wikipedia tool to find information.
Return a structured summary of what you found, citing the source."""

    # Questo worker fa una ricerca diretta (semplificato: senza loop di agente)
    resultado_wiki = buscar_wikipedia(subtarea, sentences=5)
    return completar(
        [{"role": "user", "content": f"Summarize this for: {subtarea}\n\n{resultado_wiki}"}],
        system=system, max_tokens=300
    )


def worker_analista(subtarea: str) -> str:
    """
    Worker specializzato in analisi e calcoli.
    Può fare calcoli matematici e analisi comparative.
    """
    system = """You are a data analyst. Your job is to analyze information and perform calculations.
Use the calcular tool for any mathematical operations.
Return a structured analysis with numbers and clear reasoning."""

    # Cerca di estrarre ed eseguire calcoli se il compito lo richiede
    return completar(
        [{"role": "user", "content": f"Analyze this: {subtarea}\n\nDate for reference: {obtener_fecha()}"}],
        system=system, max_tokens=300
    )


def worker_redactor(subtarea: str) -> str:
    """
    Worker specializzato nella redazione di contenuti strutturati.
    Prende informazioni grezze e le trasforma in testo chiaro e ben formattato.
    """
    system = """You are a technical writer. Your job is to produce clear, structured content.
Format your output with headers and bullet points.
Be concise, avoid repetition, and make the content actionable."""

    return completar(
        [{"role": "user", "content": subtarea}],
        system=system, max_tokens=400
    )


WORKERS: dict[str, Callable] = {
    "investigador": worker_investigador,
    "analista":     worker_analista,
    "redactor":     worker_redactor,
}

WORKERS_DESC = {
    "investigador": "Finds factual background information from reliable sources",
    "analista":     "Performs analysis, comparisons, and calculations",
    "redactor":     "Writes structured, clear content from raw information",
}


# ── Il supervisore ──────────────────────────────────────────────────────────

SYSTEM_SUPERVISOR = """You are a supervisor managing a team of specialized AI workers.
Your team: investigador (research), analista (analysis/calculations), redactor (writing).
For each round, decide: delegate to a worker OR produce the final answer.
Always end with the redactor to produce a polished final response."""


def supervisor_decide(
    objetivo: str,
    delegaciones: list[Delegacion]
) -> dict:
    """
    Il supervisore decide il passo successivo: delegare o rispondere.
    Restituisce {"accion": "delegar"|"responder", "worker": "...", "subtarea": "..."}
    """
    historial = [{"ronda": d.ronda, "worker": d.worker,
                  "subtarea": d.subtarea, "resultado": d.resultado[:150]}
                 for d in delegaciones]

    workers_str = json.dumps(WORKERS_DESC)
    prompt = f"""Objective: {objetivo}

Work done so far:
{json.dumps(historial, indent=2)}

Available workers: {workers_str}

What to do next?
- If more information or analysis is needed: delegate to a worker
- If we have enough to produce a final answer: respond directly

Return JSON: {{"accion": "delegar"|"responder", "worker": "investigador"|"analista"|"redactor"|null, "subtarea": "specific task for the worker"|null, "razon": "why this decision"}}"""

    return completar_json([{"role": "user", "content": prompt}], system=SYSTEM_SUPERVISOR)


def ejecutar_multiagente(
    objetivo: str,
    max_rondas: int = 5,
    verbose: bool = True
) -> tuple[list[Delegacion], str]:
    """
    Sistema multi-agente Supervisor + Workers.
    Il supervisore orchestra fino a max_rondas round di delega.
    """
    delegaciones: list[Delegacion] = []
    t_inicio = time.perf_counter()

    if verbose:
        print(f"\nSistema multi-agente")
        print(f"Obiettivo: {objetivo}")
        print("=" * 60)

    for ronda in range(1, max_rondas + 1):
        decision = supervisor_decide(objetivo, delegaciones)

        if decision["accion"] == "responder" or decision.get("worker") is None:
            if verbose:
                print(f"\n[Round {ronda}] Supervisore: risposta finale")
            break

        worker_nombre = decision["worker"]
        subtarea      = decision["subtarea"]
        razon         = decision.get("razon", "")

        if verbose:
            print(f"\n[Round {ronda}] Supervisore → {worker_nombre}")
            print(f"  Compito: {subtarea[:70]}")
            print(f"  Motivo:  {razon[:60]}")

        if worker_nombre not in WORKERS:
            if verbose:
                print(f"  Il worker '{worker_nombre}' non esiste. Salto.")
            continue

        t_worker = time.perf_counter()
        resultado = WORKERS[worker_nombre](subtarea)
        latencia  = (time.perf_counter() - t_worker) * 1000

        delegaciones.append(Delegacion(
            ronda=ronda, worker=worker_nombre,
            subtarea=subtarea, resultado=resultado,
            latencia_ms=round(latencia, 1)
        ))

        if verbose:
            print(f"  Risultato: {resultado[:120]}... ({latencia:.0f}ms)")

    # Sintesi finale del redattore
    contexto_final = "\n\n".join(
        f"[{d.worker}, round {d.ronda}]\n{d.resultado}" for d in delegaciones
    )
    respuesta_final = worker_redactor(
        f"Produce a final report for this objective: {objetivo}\n\nResearch gathered:\n{contexto_final}"
    )

    latencia_total = (time.perf_counter() - t_inicio) * 1000
    if verbose:
        print(f"\nTotale: {len(delegaciones)} deleghe in {latencia_total:.0f}ms")

    return delegaciones, respuesta_final


# ── Test del sistema multi-agente ─────────────────────────────────────────────
OBJETIVO_MULTIAGENTE = (
    "Write a brief technology assessment: Is FastAPI a good choice for "
    "building AI-powered APIs in 2024? Include history, current state, and recommendation."
)

delegaciones, informe = ejecutar_multiagente(OBJETIVO_MULTIAGENTE, max_rondas=4, verbose=True)

print(f"\n{'='*60}")
print("REPORT FINALE")
print(f"{'='*60}")
print(informe)

**Osservazioni** (modifica questa cella):

- Il supervisore ha distribuito bene i compiti tra i worker?
- Quanti round sono stati necessari? Pensi che fossero tutti necessari?
- Confronta la latenza del multi-agente con il Plan-and-Execute. Quando useresti l'uno e quando l'altro?

---
## Sfida finale — Pipeline di intelligence competitiva

Combina Plan-and-Execute con il sistema multi-agente per produrre
un report di analisi competitiva tra due tecnologie.
Il pianificatore struttura il lavoro; il sistema multi-agente lo esegue.

Leggi lo scheletro e completa i `# TODO`.

In [ ]:
@dataclass
class InformeCompetitivo:
    tecnologia_a:  str
    tecnologia_b:  str
    plan:          Plan | None
    delegaciones:  list[Delegacion]
    informe_final: str | None
    latencia_ms:   float

    def mostrar_resumen(self) -> None:
        print(f"\nReport: {self.tecnologia_a} vs {self.tecnologia_b}")
        print(f"Passi eseguiti:  {len(self.plan.pasos) if self.plan else 0}")
        print(f"Deleghe:         {len(self.delegaciones)}")
        print(f"Latenza totale:  {self.latencia_ms:.0f}ms")
        if self.informe_final:
            print(f"\nREPORT:\n{'='*60}")
            print(self.informe_final)


def analisis_competitivo(
    tecnologia_a: str,
    tecnologia_b: str,
    verbose: bool = True
) -> InformeCompetitivo:
    """
    Pipeline completa di analisi competitiva tra due tecnologie.

    TODO: implementa la pipeline in due fasi:

    FASE 1 — Pianificazione:
      Crea un piano per raccogliere informazioni su entrambe le tecnologie.
      L'obiettivo del piano deve includere:
      - Storia e origine di ciascuna tecnologia
      - Principali casi d'uso
      - Differenze chiave
      Usa crear_plan() con un obiettivo ben formulato.

    FASE 2 — Esecuzione multi-agente:
      Per ogni passo del piano che implica ricerca,
      delega al worker_investigador.
      Per il passo di sintesi finale, delega al worker_redactor.

      SEMPLIFICAZIONE ACCETTABILE: invece di integrare il piano con il
      sistema multi-agente (che richiede molto glue code), puoi:
      1. Eseguire ejecutar_plan() per raccogliere informazioni
      2. Poi passare il contesto completo al worker_redactor per il report

    Suggerimento: l'obiettivo del piano può essere:
      f"Research {tecnologia_a} and {tecnologia_b}: history, use cases, strengths, weaknesses."
    """
    t_inicio = time.perf_counter()

    # TODO: implementa le due fasi
    raise NotImplementedError("Implementa analisis_competitivo()")


# ── Test della pipeline ───────────────────────────────────────────────────────
# Decommenta quando hai implementato analisis_competitivo()

# COMPARACIONES = [
#     ("FastAPI",      "Django REST Framework"),
#     ("PostgreSQL",   "MongoDB"),
#     ("Docker",       "Kubernetes"),
# ]

# for tech_a, tech_b in COMPARACIONES[:1]:   # inizia con un solo confronto
#     informe = analisis_competitivo(tech_a, tech_b, verbose=True)
#     informe.mostrar_resumen()

### Estensioni opzionali

1. **Esecuzione parallela reale dei passi indipendenti:** in `ejecutar_plan()`, invece di eseguire i `pasos_listos` in serie, usa `concurrent.futures.ThreadPoolExecutor` per eseguirli in parallelo. Misura la riduzione di latenza.

2. **Worker con memoria:** dai al `worker_investigador` l'accesso alla collezione Chroma del notebook 07 (`rag_docs`). Se l'informazione è nel corpus interno, la usa; altrimenti ricorre a Wikipedia.

3. **Esportazione del report in Markdown:** dopo `analisis_competitivo()`, salva il report in un file `.md` con frontmatter (data, tecnologie confrontate, numero di fonti). Sarà la base del Progetto 2 del blocco.

In [ ]:
# Estensione 1: parallelizzazione con ThreadPoolExecutor
# Decommenta e completa per eseguire passi indipendenti in parallelo

# from concurrent.futures import ThreadPoolExecutor, as_completed

# def ejecutar_plan_paralelo(plan: Plan, verbose: bool = True) -> tuple[Plan, str]:
#     """
#     Versione parallela di ejecutar_plan().
#     I passi senza dipendenze reciproche vengono eseguiti in parallelo.
#     """
#     plan.estado = "en_ejecucion"
#     contexto    = {}
#     t_inicio    = time.perf_counter()

#     while not plan.esta_completo():
#         listos = plan.pasos_listos()
#         if not listos:
#             break

#         # TODO: esegui tutti i passi pronti in parallelo
#         # con ThreadPoolExecutor
#         ...

#     # TODO: sintesi finale
#     ...

---
## Autovalutazione della settimana

1. **Cosa ha funzionato bene?**
   ...

2. **Cosa non hai capito del tutto?**
   ...

3. **Cosa cambieresti la settimana prossima?**
   ...

4. **Tempo reale dedicato:**
   ...

---
## Prossima settimana

La settimana 12 è il **Progetto 1 del portfolio**: un RAG chatbot completo
con agente, memoria e valutazione automatica su un corpus reale del tuo dominio
(agri-food, biotech o forestale). Sarà il tuo primo elemento di portfolio per i clienti.

---
*Notebook 11 — Settimana 11 del Piano di Formazione AI Engineering*